In [50]:
import pandas as pd
from srai.regionalizers import H3Regionalizer, geocode_to_region_gdf
from srai.embedders import CountEmbedder
from srai.joiners import IntersectionJoiner
from srai.loaders import OSMOnlineLoader

In [52]:
area = geocode_to_region_gdf("Wrocław, Poland")
regionalizer = H3Regionalizer(resolution=8) # https://h3geo.org/docs/core-library/restable/
regions = regionalizer.transform(area)
regions.head()

,geometry
region_id,
881e2042c9fffff,"POLYGON ((16.98299 51.17688, 16.98808 51.17367..."
881e2042c5fffff,"POLYGON ((16.98834 51.16190, 16.99075 51.16618..."
881e2040c3fffff,"POLYGON ((17.01939 51.11909, 17.01697 51.11481..."
881e205525fffff,"POLYGON ((16.96287 51.17795, 16.97038 51.17902..."
881e20423dfffff,"POLYGON ((16.93337 51.15014, 16.93578 51.15442..."


In [53]:
from srai.plotting import plot_regions

query = {"traffic_signals" : {"highway": "traffic_signals"},
         "crossing" : {"highway": "crossing"},
         "sidewalk" : {"footway":"sidewalk"},
         "traffic_calming" : {"traffic_calming":"yes"},
         "street_lamp" : {"highway": "street_lamp"},
         "bicycle_road" : {"bicycle_road": "yes"}
}
area = geocode_to_region_gdf("Wrocław, Poland")
loader = OSMOnlineLoader()

feature_gdf = loader.load(area, query)
feature_gdf

  0%|          | 0/6 [00:00<?, ?it/s]

Grouping features: 100%|██████████| 12477/12477 [00:00<00:00, 19810.55it/s]


,geometry,traffic_signals,crossing,sidewalk,traffic_calming,street_lamp,bicycle_road
feature_id,,,,,,,
node/150597406,POINT (16.97786 51.09238),NaN,highway=crossing,NaN,NaN,NaN,NaN
node/151334674,POINT (16.97939 51.09485),highway=traffic_signals,NaN,NaN,NaN,NaN,NaN
node/153568043,POINT (16.96694 51.07828),NaN,highway=crossing,NaN,NaN,NaN,NaN
node/158719856,POINT (17.03100 51.09441),NaN,highway=crossing,NaN,NaN,NaN,NaN
node/158719876,POINT (17.04118 51.09612),highway=traffic_signals,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
way/1206982177,"LINESTRING (17.04150 51.06451, 17.04145 51.064...",NaN,NaN,footway=sidewalk,NaN,NaN,NaN
way/1207942937,"LINESTRING (16.90795 51.14688, 16.90785 51.146...",NaN,NaN,footway=sidewalk,NaN,NaN,NaN
way/1208372940,"LINESTRING (17.03943 51.06481, 17.03944 51.064...",NaN,NaN,footway=sidewalk,NaN,NaN,NaN


In [54]:
regions = regionalizer.transform(area)
joiner = IntersectionJoiner()
joint_features = joiner.transform(regions, feature_gdf)
embedder = CountEmbedder()
embeddings = embedder.transform(regions, feature_gdf, joint_features)
embeddings_feature_gdf = pd.merge(regions, embeddings, on='region_id', how='inner')
embeddings_feature_gdf

,geometry,traffic_signals_highway=traffic_signals,crossing_highway=crossing,sidewalk_footway=sidewalk,traffic_calming_traffic_calming=yes,street_lamp_highway=street_lamp,bicycle_road_bicycle_road=yes
region_id,,,,,,,
881e2042c9fffff,"POLYGON ((16.98299 51.17688, 16.98808 51.17367...",0,9,27,0,0,0
881e2042c5fffff,"POLYGON ((16.98834 51.16190, 16.99075 51.16618...",0,0,0,0,0,0
881e2040c3fffff,"POLYGON ((17.01939 51.11909, 17.01697 51.11481...",20,55,19,0,25,0
881e205525fffff,"POLYGON ((16.96287 51.17795, 16.97038 51.17902...",0,0,0,0,0,0
881e20423dfffff,"POLYGON ((16.93337 51.15014, 16.93578 51.15442...",0,13,4,0,35,0
...,...,...,...,...,...,...,...
881e204317fffff,"POLYGON ((16.89639 51.12125, 16.89880 51.12553...",0,6,1,0,0,0
881e2040e9fffff,"POLYGON ((16.98695 51.11055, 16.98454 51.10627...",0,14,0,0,0,0
881e20422dfffff,"POLYGON ((16.89557 51.15655, 16.88807 51.15548...",0,23,7,0,0,0


In [55]:
import pandas as pd


df_accidents = pd.read_csv('data/embeddings.csv')
df_accidents

,region_id,rok_2023
0,881e204059fffff,14
1,881e205561fffff,0
2,881e2041ddfffff,3
3,881e204225fffff,10
4,881e2050a1fffff,0
...,...,...
417,881e20473bfffff,0
418,881e20473dfffff,0
419,881e204509fffff,1
420,881e204545fffff,0


In [56]:
gdf = pd.merge(embeddings_feature_gdf, df_accidents, on='region_id', how='inner')
gdf

,region_id,geometry,traffic_signals_highway=traffic_signals,crossing_highway=crossing,sidewalk_footway=sidewalk,traffic_calming_traffic_calming=yes,street_lamp_highway=street_lamp,bicycle_road_bicycle_road=yes,rok_2023
0,881e2042c9fffff,"POLYGON ((16.98299 51.17688, 16.98808 51.17367...",0,9,27,0,0,0,0
1,881e2042c5fffff,"POLYGON ((16.98834 51.16190, 16.99075 51.16618...",0,0,0,0,0,0,1
2,881e2040c3fffff,"POLYGON ((17.01939 51.11909, 17.01697 51.11481...",20,55,19,0,25,0,57
3,881e205525fffff,"POLYGON ((16.96287 51.17795, 16.97038 51.17902...",0,0,0,0,0,0,0
4,881e20423dfffff,"POLYGON ((16.93337 51.15014, 16.93578 51.15442...",0,13,4,0,35,0,14
...,...,...,...,...,...,...,...,...,...
417,881e204317fffff,"POLYGON ((16.89639 51.12125, 16.89880 51.12553...",0,6,1,0,0,0,0
418,881e2040e9fffff,"POLYGON ((16.98695 51.11055, 16.98454 51.10627...",0,14,0,0,0,0,1
419,881e20422dfffff,"POLYGON ((16.89557 51.15655, 16.88807 51.15548...",0,23,7,0,0,0,5
420,881e2050a1fffff,"POLYGON ((16.87946 51.20145, 16.88455 51.19824...",0,0,0,0,0,0,0


In [155]:
bins = [-1, 0, 3, 10, float("inf")]  # Bins: (-1, 0], (0, 3], (3, 10], (10, inf)
label_names = ['no risk', 'low risk', 'medium risk', 'high risk']
gdf['risk_level'] = pd.cut(gdf['rok_2023'], bins=bins, labels=label_names, right=False)
gdf

,region_id,geometry,traffic_signals_highway=traffic_signals,crossing_highway=crossing,sidewalk_footway=sidewalk,traffic_calming_traffic_calming=yes,street_lamp_highway=street_lamp,bicycle_road_bicycle_road=yes,rok_2023,risk_level
0,881e2042c9fffff,"POLYGON ((16.98299 51.17688, 16.98808 51.17367...",0,9,27,0,0,0,0,low risk
1,881e2042c5fffff,"POLYGON ((16.98834 51.16190, 16.99075 51.16618...",0,0,0,0,0,0,1,low risk
2,881e2040c3fffff,"POLYGON ((17.01939 51.11909, 17.01697 51.11481...",20,55,19,0,25,0,57,high risk
3,881e205525fffff,"POLYGON ((16.96287 51.17795, 16.97038 51.17902...",0,0,0,0,0,0,0,low risk
4,881e20423dfffff,"POLYGON ((16.93337 51.15014, 16.93578 51.15442...",0,13,4,0,35,0,14,high risk
...,...,...,...,...,...,...,...,...,...,...
417,881e204317fffff,"POLYGON ((16.89639 51.12125, 16.89880 51.12553...",0,6,1,0,0,0,0,low risk
418,881e2040e9fffff,"POLYGON ((16.98695 51.11055, 16.98454 51.10627...",0,14,0,0,0,0,1,low risk
419,881e20422dfffff,"POLYGON ((16.89557 51.15655, 16.88807 51.15548...",0,23,7,0,0,0,5,medium risk
420,881e2050a1fffff,"POLYGON ((16.87946 51.20145, 16.88455 51.19824...",0,0,0,0,0,0,0,low risk


In [156]:
import networkx as nx


def create_graph(weights=None):
    graph = nx.Graph()
    for index, row in gdf.iterrows():
        current_polygon = row['geometry']
        node_data = {
            'geometry': current_polygon,
            'traffic_signals_highway=traffic_signals': row['traffic_signals_highway=traffic_signals'],
            'crossing_highway=crossing': row['crossing_highway=crossing'],
            'sidewalk_footway=sidewalk': row['sidewalk_footway=sidewalk'],
            'traffic_calming_traffic_calming=yes': row['traffic_calming_traffic_calming=yes'],
            'street_lamp_highway=street_lamp': row['street_lamp_highway=street_lamp'],
            'bicycle_road_bicycle_road=yes': row['bicycle_road_bicycle_road=yes'],
            'risk_of_accident_level': row['risk_level']
        }
        graph.add_node(index, **node_data)

    for i in range(len(gdf)):
        for j in range(i + 1, len(gdf)):
            node_i = gdf.iloc[i]
            node_j = gdf.iloc[j]

            if node_i['geometry'].touches(node_j['geometry']):
                weight = 1
            else:
                try:
                    weight = nx.shortest_path_length(graph, i, j)
                except nx.NetworkXNoPath:
                    continue  # Skip this pair of nodes with no path

            graph.add_edge(i, j, weight=weight)
    return graph

In [157]:
graph = create_graph()

In [158]:
from sklearn.preprocessing import LabelEncoder


labels = gdf['risk_level']

label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)

labels_encoded

array([1, 1, 0, 1, 0, 0, 1, 2, 1, 1, 2, 0, 1, 2, 2, 0, 0, 2, 2, 1, 1, 2,
       0, 1, 1, 1, 1, 2, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 0,
       1, 1, 2, 0, 0, 1, 1, 1, 1, 1, 2, 0, 1, 2, 1, 2, 1, 2, 1, 0, 1, 0,
       2, 2, 1, 0, 1, 2, 1, 1, 0, 1, 2, 1, 2, 2, 0, 1, 2, 0, 0, 1, 1, 1,
       0, 1, 1, 2, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1,
       2, 0, 1, 1, 1, 0, 0, 2, 0, 2, 1, 0, 1, 2, 1, 1, 2, 0, 1, 2, 1, 1,
       2, 1, 0, 0, 1, 2, 1, 1, 2, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 2, 1,
       0, 0, 1, 2, 0, 1, 2, 0, 1, 1, 1, 0, 2, 1, 2, 1, 0, 1, 1, 0, 2, 1,
       1, 2, 2, 1, 1, 1, 0, 0, 1, 0, 1, 1, 2, 1, 0, 0, 1, 2, 1, 2, 0, 0,
       2, 1, 1, 0, 0, 1, 1, 2, 2, 1, 2, 1, 0, 1, 1, 0, 1, 1, 2, 1, 1, 1,
       2, 1, 1, 2, 2, 1, 1, 1, 1, 1, 1, 0, 1, 2, 0, 1, 1, 0, 0, 2, 2, 0,
       2, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 2, 0,
       1, 1, 1, 2, 1, 0, 0, 0, 2, 2, 2, 1, 2, 1, 1, 2, 1, 1, 1, 1, 2, 0,
       0, 0, 1, 1, 0, 1, 0, 2, 2, 1, 1, 1, 1, 2, 1,

In [162]:
import torch
from torch_geometric.data import Data
import numpy as np


feature_columns = [
    'traffic_signals_highway=traffic_signals',
    'crossing_highway=crossing',
    'sidewalk_footway=sidewalk',
    'traffic_calming_traffic_calming=yes',
    'street_lamp_highway=street_lamp',
    'bicycle_road_bicycle_road=yes'
]

node_features = gdf[feature_columns].to_numpy(dtype=np.float32)

labels = labels_encoded


edge_index = graph.edges()
x = torch.tensor(node_features, dtype=torch.float)
y = torch.tensor(labels, dtype=torch.long)

data = Data(x=x, edge_index=edge_index, y=y)

In [225]:
 #pytorch geo

----

In [150]:
from torch import nn
from torch_geometric.nn import GCNConv


class GCNModel(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.act1 = nn.ReLU()
        self.conv2 = GCNConv(hidden_dim, out_dim)
        self.act2 = nn.ReLU()

    def forward(self, x, edge_index):
        x, edge_index = data.x, data.edge_index
        x = self.act1(self.conv1(x, edge_index))
        x = self.act2(self.conv2(x, edge_index))
        return x

In [151]:
import pytorch_lightning as pl
import torch
from sklearn.metrics import roc_auc_score
from torch import nn
from torch_geometric.data import Data

class SupervisedNodeClassificationGNN(pl.LightningModule):
    def __init__(self, gnn: nn.Module, emb_dim: int, num_classes: int):
        super().__init__()
        self.gnn = gnn
        self.classification_head = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(emb_dim, num_classes),
            nn.LogSoftmax(dim=1),
        )
        self.loss_fn = nn.NLLLoss()

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
    ) -> torch.Tensor:
        return self.gnn(x, edge_index)

    def training_step(self, batch):
        x, edge_index, y = batch.x, batch.edge_index, batch.y
        y_pred = self(x, edge_index)
        loss = self.loss_fn(y_pred, y)
        return loss

    def validation_step(self, batch):
        x, edge_index, y = batch.x, batch.edge_index, batch.y
        y_pred = self(x, edge_index)
        auc = self.calculate_auc(y_pred, y)
        self.log("val_auc", auc)
        return {"val_auc": auc}

    def test_step(self, batch):
        x, edge_index, y = batch.x, batch.edge_index, batch.y
        y_pred = self(x, edge_index)
        auc = self.calculate_auc(y_pred, y)
        self.log("test_auc", auc)
        return {"test_auc": auc}

    def configure_optimizers(self):
        return torch.optim.AdamW(
            params=self.parameters(),
            lr=1e-3,
            weight_decay=5e-4,
        )

    def calculate_auc(self, y_pred, y):
        # Assuming y_pred is of shape (N, num_classes) and y is of shape (N,)
        y_prob = y_pred.exp()  # Convert log probabilities to probabilities
        auc = roc_auc_score(y_true=y, y_score=y_prob, multi_class="ovr")
        return auc


In [169]:
num_features = data.x.shape[1]
hidden_dim = 256
out_dim = 128
num_classes = len(data.y.unique())

gnn = GCNModel(in_dim=num_features, hidden_dim=hidden_dim, out_dim=out_dim)
model = SupervisedNodeClassificationGNN(gnn=gnn, emb_dim=out_dim, num_classes=num_classes)


In [221]:
from torch_geometric.data import DataLoader, Data
from torch_geometric.utils import mask_select
from sklearn.model_selection import train_test_split

# Zakryj 20% wierzchołków jako zbiór treningowy
train_nodes, test_nodes = train_test_split(range(len(data.y)), test_size=0.2, random_state=42)

train_masked_edges = []
test_masked_edges = []
for (edge1, edge2) in data.edge_index:
    if edge1 in train_nodes and edge2 in train_nodes:
        train_masked_edges.append((edge1, edge2))

train_data = Data(
    x=data.x[train_nodes],
    edge_index=train_masked_edges,
    y=data.y[train_nodes]
)

test_data = Data(
    x=data.x[test_nodes],
    edge_index=data.edge_index,
    y=data.y[test_nodes]
)



# Twórz DataLoader dla danych treningowych i testowych
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

# Trenuj model
trainer = pl.Trainer(max_epochs=10)  # Dostosuj liczbę epok do swoich potrzeb
trainer.fit(model, train_loader)

# # Oceń model na danych testowych
# results = trainer.test(model, test_loader)

# # Wydrukuj wyniki
# print(results)


Training: |          | 0/? [00:00<?, ?it/s]

KeyError: 0